Erik de KlerkProject partner LF bandpass filter

# 1. Introduction

## 1.1 Motivation
Maintaining alertness is essential in occupations where lapses in attention can lead to serious consequences, such as long-distance truck driving and air traffic control. Detecting drowsiness before it leads to impaired performance is therefore an important safety challenge. 

Several approaches have been proposed to monitor fatigue, including behavioural observations and physiological measurements. Among these methods, analysing a person’s electrocardiogram (ECG) provides a reliable and non-intrusive way to continuously monitor physiological changes related to fatigue.

## 1.2 Physiological Basis of Drowsiness Detection
Changes in alertness are closely linked to the activity of the autonomic nervous system. According to Fujiwara et al. (2019), when a person transitions from an awake state to the first stage of sleep (N1), the balance between the sympathetic and parasympathetic nervous systems shifts. During wakefulness, sympathetic activity dominates, whereas parasympathetic activity becomes more prominent as the person becomes drowsy.

These changes can be quantified using **Heart Rate Variability (HRV)**, which measures the variation in time between consecutive R-peaks of the ECG signal. These intervals, referred to as **R–R intervals (RRI)**, contain useful information about autonomic nervous system activity.

When the HRV signal is analysed in the frequency domain, two important spectral bands can be identified:

* **Low-Frequency (LF) Band ($0.04 \text{ Hz} - 0.15 \text{ Hz}$):** Represents a combination of sympathetic and parasympathetic activity.
* **High-Frequency (HF) Band ($0.15 \text{ Hz} - 0.40 \text{ Hz}$):** Mainly associated with parasympathetic activity.

By comparing the power contained in these two bands using the **LF/HF ratio**, it becomes possible to observe shifts in autonomic balance. A decrease in this ratio indicates stronger parasympathetic influence, which can be interpreted as a sign that the individual is becoming drowsy.

## 1.3 Signal Processing Objective
The goal of this assignment is to estimate a person's level of drowsiness by analysing the frequency content of their ECG-derived HRV signal. To achieve this, digital filters are used to isolate the LF and HF frequency bands of interest.

Specifically, **Infinite Impulse Response (IIR) digital bandpass filters** will be designed to extract the LF and HF components of the signal. The power contained within these filtered signals will then be used to compute the LF/HF ratio, which serves as an indicator of the subject's physiological state.

## 1.4 Implementation Constraints
Several restrictions are imposed on the filter design process to ensure that the filters are derived from first principles:

1. **Manual Filter Design:** The IIR filters must be derived mathematically and implemented using fundamental signal processing techniques.
2. **Restricted DSP Functions:** Standard automatic filter design functions such as `scipy.signal.butter()`, `scipy.signal.iirdesign`, and `scipy.signal.iirfilter` may not be used.
3. **Experimental Validation:** The implemented filters will be applied to a real ECG dataset (`ECGdata.csv`). The resulting LF/HF ratios will be plotted and compared with the annotated sleepiness levels provided in the dataset.

# 2. Input Signal Analysis

Before implementing the digital filters, the ECG signal must first be examined to understand the characteristics of the data and the features that will be extracted for analysis. Both the temporal structure of the signal and the sampling requirements influence how the subsequent processing steps are designed.

## 2.1 Time-Domain Characteristics and Preprocessing
In the time domain, the most prominent feature of the ECG waveform is the **QRS complex**, where the **R-peak** represents the largest and most easily identifiable point in each heartbeat cycle. By measuring the time difference between consecutive R-peaks, the **R–R interval (RRI)** sequence can be obtained. This sequence forms the basis for Heart Rate Variability (HRV) analysis.

Reliable HRV analysis requires the ECG signal to be sampled at a sufficiently high rate to accurately detect R-peaks. Clinical recommendations indicate that a sampling frequency ($f_s$) of at least $200 \text{ Hz}$ is required to ensure precise detection of these events.

To analyse variations in heart rate over time while maintaining approximate signal stationarity, the data will be processed using a **3-minute rolling window**. This approach allows the system to continuously evaluate changes in heart rate behaviour while still capturing gradual physiological transitions.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
from scipy.interpolate import CubicSpline

# ==========================================================
# 1) Load ECG dataset
# ==========================================================
csv_name = "ECGdata.csv"
ecg_data = pd.read_csv(csv_name)

# ==========================================================
# 2) Extract time information and calculate ECG sampling rate
# ==========================================================
# Remove square brackets from the time strings
clean_time = ecg_data["Time"].str.strip("[]")

# Convert the cleaned strings to datetime format
time_stamps = pd.to_datetime(clean_time, format="%H:%M:%S.%f")

# Find the median time step between consecutive samples
sample_interval = time_stamps.diff().dt.total_seconds().median()

# Calculate sampling frequency
sampling_rate_ecg = int(round(1 / sample_interval))

print(f"ECG sampling rate (Calculated): {sampling_rate_ecg} Hz")

if sampling_rate_ecg >= 200:
 print("STATUS: Sampling-rate requirement satisfied (>= 200 Hz).")
else:
 print("STATUS: Sampling-rate requirement not satisfied.")

# ==========================================================
# 3) Select the first 3 minutes of ECG data
# ==========================================================
analysis_time_sec = 3 * 60
samples_needed = analysis_time_sec * sampling_rate_ecg

# Extract raw ECG samples for the selected clip
full_ecg_signal = ecg_data["ECG"].to_numpy()
ecg_segment = full_ecg_signal[:samples_needed]

# Create a matching time axis for the clipped signal
time_axis = np.arange(len(ecg_segment)) / sampling_rate_ecg

# ==========================================================
# 4) Detect R-peaks
# ==========================================================
# Minimum allowed distance between peaks:
# 0.5 seconds is used to avoid detecting multiple peaks
# within the same heartbeat complex
minimum_peak_spacing = int(0.5 * sampling_rate_ecg)

# Peak height threshold:
# peaks should be clearly above the average signal level
peak_height_limit = np.mean(ecg_segment) + 1.5 * np.std(ecg_segment)

# Detect peaks
r_peak_indices, peak_info = find_peaks(
 ecg_segment,
 distance=minimum_peak_spacing,
 height=peak_height_limit
)

# Convert peak sample indices to time values
r_peak_times = time_axis[r_peak_indices]

# ==========================================================
# 5) Calculate R-R intervals
# ==========================================================
# RRI is the time between consecutive R-peaks
rr_intervals_ms = np.diff(r_peak_times) * 1000 # convert to milliseconds

# Associate each RRI value with the time of the second peak
rr_time_axis = r_peak_times[1:]

print(f"Detected {len(r_peak_indices)} R-peaks in the 3-minute segment.")

# ==========================================================
# 6) Visualize ECG signal with detected R-peaks
# ==========================================================
plt.figure(figsize=(13,4))

# Plot ECG waveform
plt.plot(
 time_axis,
 ecg_segment,
 color="steelblue",
 linewidth=1.5,
 label="ECG waveform"
)

# Highlight detected R-peaks
plt.plot(
 r_peak_times,
 ecg_segment[r_peak_indices],
 color="red",
 marker="o",
 linestyle="",
 markersize=5,
 zorder=3,
 label="Detected R-peaks"
)

# Focus on the first 15 seconds
plt.xlim(0, 15)

# Labels and title
plt.title("ECG Segment with Detected Heartbeat Peaks")
plt.xlabel("Time [seconds]")
plt.ylabel("Signal Amplitude (mV)")

# grid appearance
plt.grid(True, linestyle="--", linewidth=0.5, alpha=0.6)

plt.legend(loc="upper right")
plt.show()


## 2.2 Frequency-Domain Analysis (FFT and Power Spectral Density)

The R–R interval (RRI) sequence obtained from the ECG signal is inherently unevenly sampled, since heartbeats do not occur at perfectly uniform time intervals. Spectral analysis methods such as the Fast Fourier Transform (FFT) require evenly spaced samples, therefore the RRI sequence must first be interpolated and resampled.

To achieve this, the RRI data are interpolated using a **cubic spline** and then resampled at a uniform sampling rate of $4 \text{ Hz}$. This produces a regularly spaced time series suitable for frequency-domain analysis.

Before computing the FFT, the mean of the signal is removed to eliminate any **DC component** that could dominate the spectrum and obscure the lower frequency content. The FFT is then applied to estimate the **Power Spectral Density (PSD)** of the HRV signal.

From the resulting PSD, the spectral power within the two frequency bands of interest is extracted:

* **Low Frequency (LF):** $0.04\text{ Hz} \le f < 0.15\text{ Hz}$
* **High Frequency (HF):** $0.15\text{ Hz} \le f \le 0.40\text{ Hz}$

These bands correspond to the frequency ranges that will later be isolated using the designed digital filters.

In [ ]:
# ==========================================================
# 1) Interpolate the R-R intervals and resample at 4 Hz
# ==========================================================
resample_rate = 4.0 # Hz

# Create a uniformly spaced time axis over the valid RRI interval
uniform_rri_time = np.arange(rr_time_axis[0], rr_time_axis[-1], 1 / resample_rate)

# Fit a cubic spline to the unevenly sampled RRI data
rri_spline = CubicSpline(rr_time_axis, rr_intervals_ms)

# Evaluate the spline on the new uniform time axis
rri_uniform = rri_spline(uniform_rri_time)

# Remove the mean to suppress the DC component before FFT
rri_zero_mean = rri_uniform - np.mean(rri_uniform)

# ==========================================================
# 2) Compute FFT and estimate the power spectral density
# ==========================================================
num_points = len(rri_zero_mean)

frequency_axis = np.fft.rfftfreq(num_points, d=1 / resample_rate)
fft_result = np.fft.rfft(rri_zero_mean)

# Estimate PSD from FFT magnitude
power_spectral_density = (np.abs(fft_result) ** 2) / (num_points * resample_rate)

# ==========================================================
# 3) Isolate LF and HF bands and calculate band powers
# ==========================================================
lf_band = (frequency_axis >= 0.04) & (frequency_axis < 0.15)
hf_band = (frequency_axis >= 0.15) & (frequency_axis <= 0.40)

frequency_resolution = frequency_axis[1] - frequency_axis[0]

lf_power = np.sum(power_spectral_density[lf_band]) * frequency_resolution
hf_power = np.sum(power_spectral_density[hf_band]) * frequency_resolution
lf_hf_ratio = lf_power / hf_power

print(f"LF band power from FFT (0.04-0.15 Hz): {lf_power:.2f} ms^2")
print(f"HF band power from FFT (0.15-0.40 Hz): {hf_power:.2f} ms^2")
print(f"Reference LF/HF ratio: {lf_hf_ratio:.2f}")

# ==========================================================
# 4) Plot the power spectral density
# ==========================================================
plt.figure(figsize=(11, 5))

# Main PSD curve
plt.plot(
 frequency_axis,
 power_spectral_density,
 linewidth=1.8,
 color="navy",
 label="RRI power spectrum"
)

# Highlight LF band
plt.fill_between(
 frequency_axis,
 power_spectral_density,
 where=lf_band,
 color="goldenrod",
 alpha=0.45,
 label="LF band"
)

# Highlight HF band
plt.fill_between(
 frequency_axis,
 power_spectral_density,
 where=hf_band,
 color="seagreen",
 alpha=0.35,
 label="HF band"
)

# Add band edge markers for clearer interpretation
plt.axvline(0.04, color="goldenrod", linestyle="--", linewidth=1)
plt.axvline(0.15, color="gray", linestyle="--", linewidth=1)
plt.axvline(0.40, color="seagreen", linestyle="--", linewidth=1)

plt.title("Power Spectral Density of Uniformly Resampled RRI")
plt.xlabel("Frequency [Hz]")
plt.ylabel("PSD [ms$^2$/Hz]")
plt.xlim(0, 0.5)
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend(loc="upper center", ncol=3)
plt.tight_layout()
plt.show()

# 3. IIR Filter Design Methodology

## 3.1 Selection of Filter Type

In order to isolate the Low-Frequency (LF) and High-Frequency (HF) components of the HRV signal, an appropriate digital filter structure must be selected. The chosen filter should accurately separate the desired frequency bands without introducing distortions that could influence the physiological interpretation of the results.

Two common IIR filter families were considered:

* **Chebyshev Filters:** These filters provide a sharper transition between passband and stopband, resulting in a steeper roll-off. However, they introduce ripple within the passband, which can artificially modify the amplitude of certain frequency components. For HRV analysis, this behaviour could distort the calculated LF/HF power ratio.

* **Butterworth Filters:** Butterworth filters are characterized by a **maximally flat passband**, meaning that no ripple is present within the passband region. Although the transition between passband and stopband is more gradual, the amplitude response remains smooth and stable across the frequencies of interest.

Since accurate amplitude preservation is important for reliable HRV power estimation, a **Butterworth filter** is selected for this design.

To maintain numerical stability and simplify implementation using a direct-form difference equation, a prototype filter order of **$N = 2$** is chosen. After applying the bandpass transformation, this results in a **4th-order digital bandpass filter**.

---

## 3.2 Mathematical Design Procedure

The filter coefficients are derived using fundamental signal processing principles rather than automated filter design tools. The overall design process follows several key steps.

### Pre-warping of Frequency Edges
The bilinear transform introduces frequency distortion between the analog and digital domains. To compensate for this effect, the digital cutoff frequencies ($f_1$, $f_2$) are first converted to their equivalent analog frequencies ($\Omega_1$, $\Omega_2$) using the pre-warping relation

$$
\Omega = \frac{2}{T} \tan\left(\frac{2\pi f T}{2}\right)
$$

where $T$ is the sampling period. 
From these values, the center frequency and bandwidth of the analog bandpass filter are determined as

$$
\Omega_0 = \sqrt{\Omega_1 \Omega_2}
$$

$$
B = \Omega_2 - \Omega_1
$$

---

### Analog Lowpass Prototype
The design begins with a normalized analog lowpass Butterworth prototype. 
For a filter of order $N$, the poles are evenly distributed along a semicircle in the left half of the complex $s$-plane and are given by

$$
p_k = e^{j\pi \frac{2k + N + 1}{2N}}, \quad k = 0,1,\dots,N-1
$$

These poles define the denominator of the normalized lowpass transfer function.

---

### Lowpass-to-Bandpass Transformation
To obtain a bandpass response, the lowpass prototype is transformed using the substitution

$$
s \rightarrow \frac{s^2 + \Omega_0^2}{B s}
$$

This transformation converts each lowpass pole into two bandpass poles. The resulting poles are obtained by solving the quadratic equation

$$
s^2 - p_k B s + \Omega_0^2 = 0
$$

This step produces the analog bandpass filter in the continuous-time domain.

---

### Bilinear Transform
The analog bandpass filter must then be converted into a digital filter. This is achieved using the bilinear transform, which maps the $s$-plane into the $z$-plane according to

$$
s = \frac{2}{T}\frac{z-1}{z+1}
$$

Rearranging this expression allows each analog pole $s_k$ to be mapped into its digital equivalent

$$
z_k = \frac{\frac{2}{T} + s_k}{\frac{2}{T} - s_k}
$$

This mapping ensures that stable analog poles remain inside the unit circle in the digital domain.

---

### Zero Placement and Polynomial Expansion
A digital bandpass filter requires zeros at both the DC frequency and the Nyquist frequency. For an $N$ th-order bandpass design, this results in:

* $N$ zeros at $z = 1$
* $N$ zeros at $z = -1$

After determining the pole and zero locations, the corresponding polynomials are expanded to obtain the numerator coefficients and denominator coefficients. These coefficients define the final difference equation used to implement the digital filter.

# 4. Step-by-Step Filter Design

> **Project partner:** Performed the filter design demonstration for the **LF band ($0.04 - 0.15 \text{ Hz}$)** in **Section 4.1**, where the normalized lowpass prototype is transformed into an analog bandpass filter using the corresponding pre-warped frequencies.

> **Erik de Klerk** I Performed the design procedure for the **HF band ($0.15 - 0.40 \text{ Hz}$)** in **Section 4.2**, demonstrating how the normalized lowpass prototype is mapped to an analog bandpass filter using the pre-warped frequencies associated with the HF range.

>> **Note:** The later steps of the filter design process, including the **digital transformation and stability verification**, were combined into a single function within the implementation for simplicity. This effectively merges the content into one computational step.
---

## 4.1 LF: Analog Lowpass Prototype and Bandpass Transformation

Before converting the design into the digital domain, the analog prototype filter must first be defined and analysed. This provides a reference model from which the final bandpass filter will be derived.

For a Butterworth filter of order $N = 2$, the normalized analog lowpass transfer function is given by

$$
H_{LP}(s) = \frac{1}{s^2 + \sqrt{2}s + 1}
$$

This prototype exhibits the characteristic **maximally flat magnitude response** in the passband. The normalized lowpass model will later be transformed into a bandpass filter by applying the appropriate frequency mapping using the pre-warped cutoff frequencies.

In [ ]:
import scipy.signal as signal # We only use this for signal.freqs() to draw the plot!
import numpy as np
import matplotlib.pyplot as plt

# --- 1. Analog Lowpass Prototype (N=2) ---
# H_lp(s) = 1 / (s^2 + sqrt(2)*s + 1)
b_lp_analog = [1]
a_lp_analog = [1, np.sqrt(2), 1]

# Calculate continuous-time frequency response (Plotting function is allowed)
w_lp, h_lp = signal.freqs(b_lp_analog, a_lp_analog)

# --- 2. Analog Bandpass Transformation (LF Example) via First Principles Algebra ---
fs_resample = 4.0
T = 1.0 / fs_resample

# Pre-warping
Wa1_lf = (2/T) * np.tan(2 * np.pi * 0.04 * T / 2)
Wa2_lf = (2/T) * np.tan(2 * np.pi * 0.15 * T / 2)
W0_lf = np.sqrt(Wa1_lf * Wa2_lf)
BW_lf = Wa2_lf - Wa1_lf

# Algebraic mapping of H(s) substituting s -> (s^2 + W0^2) / (BW * s)
# Numerator: BW^2 * s^2
b_bp_analog = [BW_lf**2, 0, 0]

# Denominator: s^4 + sqrt(2)*BW*s^3 + (2*W0^2 + BW^2)*s^2 + sqrt(2)*BW*W0^2*s + W0^4
a_bp_analog = [
 1, # s^4
 np.sqrt(2) * BW_lf, # s^3
 (2 * W0_lf**2) + (BW_lf**2), # s^2
 np.sqrt(2) * BW_lf * (W0_lf**2), # s^1
 W0_lf**4 # s^0
]

w_bp, h_bp = signal.freqs(b_bp_analog, a_bp_analog, worN=np.logspace(-2, 1, 1000))

# --- Plotting the Analog Transfer Functions ---
plt.figure(figsize=(12, 5))

# Plot Lowpass
plt.subplot(1, 2, 1)
plt.semilogx(w_lp, 20 * np.log10(np.abs(h_lp)), color='purple', linewidth=2)
plt.title("Analog Lowpass Prototype $H_{LP}(s)$")
plt.xlabel("Frequency (rad/s)")
plt.ylabel("Gain (dB)")
plt.axvline(1, color='red', linestyle='--', label='Cutoff (1 rad/s)')
plt.grid(True, which='both', linestyle=':')
plt.ylim(-40, 5)
plt.legend()

# Plot Bandpass
plt.subplot(1, 2, 2)
plt.semilogx(w_bp, 20 * np.log10(np.abs(h_bp)), color='orange', linewidth=2)
plt.title("Analog Bandpass Transformation $H_{BP}(s)$ [LF Band]")
plt.xlabel("Frequency (rad/s)")
plt.axvline(Wa1_lf, color='red', linestyle='--', label='Passband Edges')
plt.axvline(Wa2_lf, color='red', linestyle='--')
plt.grid(True, which='both', linestyle=':')
plt.ylim(-40, 5)
plt.legend()

plt.tight_layout()
plt.show()

## 4.2 HF: Analog Lowpass Prototype & Bandpass Mapping

In [ ]:
import scipy.signal as signal
import numpy as np
import matplotlib.pyplot as plt
import sympy as sim

# 1) Analog Lowpass Prototype (N = 2 Butterworth)
# H_lowpass(s) = 1 
# --------------------
# (s^2 + sqrt(2)s + 1)

lp_num = [1] # numerator
lp_den = [1, np.sqrt(2), 1] # denominator

freq_lp, response_lp = signal.freqs(lp_num, lp_den,worN=np.logspace(-3, 2, 2000))

# 2) Analog Bandpass Transformation (HF Band)
# HF band: 0.15 – 0.40 Hz

fs = 4.0 # sampling frequency
T = 1 / fs # sampling period

f_pass1_HF = 0.15 # lower passband edge (Hz)
f_pass2_HF = 0.40 # upper passband edge (Hz)

# Pre-warp frequencies (rad/s)
omega_pass1_HF = (2/T) * np.tan(2*np.pi*f_pass1_HF*T/2)
omega_pass2_HF = (2/T) * np.tan(2*np.pi*f_pass2_HF*T/2)

omega_center_HF = np.sqrt(omega_pass1_HF * omega_pass2_HF)
bandwidth_HF = omega_pass2_HF - omega_pass1_HF

# Lowpass → Bandpass mapping
s, u = sim.symbols("s u", complex=True)

w0 = sim.Float(omega_center_HF)
B = sim.Float(bandwidth_HF)

# Define lowpass transfer function symbolically (N=2)
H_lowpass = 1 / (s**2 + sim.sqrt(2)*s + 1)

# Apply mapping
H_bandpass = sim.simplify(H_lowpass.subs(s, (u**2 + w0**2) / (B*u)))

# Extract numerator/denominator
num_u, den_u = sim.fraction(H_bandpass)

# Expand and form polynomials in u
num_poly = sim.Poly(sim.expand(num_u), u)
den_poly = sim.Poly(sim.expand(den_u), u)

# Get coefficients (highest power first)
bp_num_sym = num_poly.all_coeffs()
bp_den_sym = den_poly.all_coeffs()

# Convert SymPy -> float arrays for SciPy
bp_num = np.array([float(c) for c in bp_num_sym], dtype=float)
bp_den = np.array([float(c) for c in bp_den_sym], dtype=float)

# Frequency response of analog bandpass
freq_bp, response_bp = signal.freqs(bp_num, bp_den, worN=np.logspace(-2, 2, 2000))

# 3) Plotting
plt.figure(figsize=(12,5))
#Lowpass Prototype 
plt.subplot(1,2,1)
plt.semilogx(freq_lp,20*np.log10(np.abs(response_lp)),linewidth=2)
plt.title("Analog Lowpass Prototype $H_{LP}(s)$")
plt.xlabel("Frequency (rad/s)")
plt.ylabel("Gain (dB)")
plt.axvline(1,linestyle='--',label='Cutoff (1 rad/s)')
plt.grid(True, which='both', linestyle=':')
plt.ylim(-40,5)
plt.legend()

#Bandpass (HF Band)
plt.subplot(1,2,2)
plt.semilogx(freq_bp,20*np.log10(np.abs(response_bp)),linewidth=2)
plt.title("Analog Bandpass Transformation $H_{BP}(s)$ [HF Band]")
plt.xlabel("Frequency (rad/s)")
plt.axvline(omega_pass1_HF,linestyle='--', label='Passband Edges')
plt.axvline(omega_pass2_HF,linestyle='--')
plt.grid(True, which='both', linestyle=':')
plt.ylim(-40,5)
plt.legend()

plt.tight_layout()
plt.show()

## 4.3 LF + HF: Digital Transformation and Stability Verification

The final stage of the filter design involves converting the analog bandpass filter into a discrete-time representation. This is achieved by applying the **Bilinear Transform**, which produces the digital filter coefficients required for implementation in the difference equation.

Once the digital filter coefficients are obtained, it is necessary to verify that the resulting system is **stable**. which is determined by examining the locations of the filter poles in the $z$-plane. 

To perform this verification, the roots of the denominator polynomial (the poles) are calculated and plotted in the complex plane. A discrete-time filter is considered stable only if all of its poles lie strictly **inside the unit circle**, meaning that their magnitudes satisfy the condition $|z| < 1$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ==========================================================
# Function: Design Butterworth bandpass filter coefficients
# using first-principles pole-zero mapping
# ==========================================================
def build_butterworth_bandpass(f_start, f_stop, sample_rate, filter_order=2):
 """
 Computes the digital numerator and denominator coefficients
 of a Butterworth bandpass filter using first-principles steps:
 1. Pre-warp digital edge frequencies
 2. Find normalized analog lowpass poles
 3. Map lowpass poles to analog bandpass poles
 4. Apply the bilinear transform
 5. Form numerator and denominator polynomials
 6. Normalize gain at the digital center frequency
 """
 
 # Sampling period
 sample_period = 1.0 / sample_rate

 # Convert cutoff frequencies from Hz to rad/s
 omega_low = 2 * np.pi * f_start
 omega_high = 2 * np.pi * f_stop

 # ------------------------------------------------------
 # 1) Pre-warp the digital band edges
 # ------------------------------------------------------
 analog_edge_low = (2 / sample_period) * np.tan(omega_low * sample_period / 2)
 analog_edge_high = (2 / sample_period) * np.tan(omega_high * sample_period / 2)

 analog_center = np.sqrt(analog_edge_low * analog_edge_high)
 analog_bandwidth = analog_edge_high - analog_edge_low

 # ------------------------------------------------------
 # 2) Determine the normalized Butterworth lowpass poles
 # ------------------------------------------------------
 lowpass_poles = []
 for pole_index in range(filter_order):
 pole = np.exp(1j * np.pi * (2 * pole_index + filter_order + 1) / (2 * filter_order))
 lowpass_poles.append(pole)
 lowpass_poles = np.array(lowpass_poles)

 # ------------------------------------------------------
 # 3) Map the lowpass poles into analog bandpass poles
 # ------------------------------------------------------
 bandpass_poles = []

 for lp_pole in lowpass_poles:
 half_bw_term = (lp_pole * analog_bandwidth) / 2
 root_term = np.sqrt(half_bw_term**2 - analog_center**2 + 0j)

 pole_a = half_bw_term + root_term
 pole_b = half_bw_term - root_term

 bandpass_poles.extend([pole_a, pole_b])

 bandpass_poles = np.array(bandpass_poles)

 # ------------------------------------------------------
 # 4) Map analog poles to digital poles using bilinear transform
 # ------------------------------------------------------
 digital_poles = ((2 / sample_period) + bandpass_poles) / ((2 / sample_period) - bandpass_poles)

 # A bandpass filter places zeros at z = 1 and z = -1
 digital_zeros = np.array(([1.0] * filter_order) + ([-1.0] * filter_order))

 # ------------------------------------------------------
 # 5) Convert zeros and poles to polynomial coefficients
 # ------------------------------------------------------
 denominator_coeffs = np.poly(digital_poles)
 numerator_coeffs = np.poly(digital_zeros)

 # ------------------------------------------------------
 # 6) Normalize the gain at the digital center frequency
 # ------------------------------------------------------
 digital_center_freq = 2 * np.arctan(analog_center * sample_period / 2)
 z_eval = np.exp(1j * digital_center_freq)

 unscaled_response = np.polyval(numerator_coeffs, z_eval) / np.polyval(denominator_coeffs, z_eval)
 gain_factor = 1.0 / np.abs(unscaled_response)

 numerator_coeffs = np.real(numerator_coeffs * gain_factor)
 denominator_coeffs = np.real(denominator_coeffs)

 return numerator_coeffs, denominator_coeffs


# ==========================================================
# Generate filter coefficients for LF and HF bands
# ==========================================================
lf_numerator, lf_denominator = build_butterworth_bandpass(
 f_start=0.04,
 f_stop=0.15,
 sample_rate=fs_resample,
 filter_order=2
)

hf_numerator, hf_denominator = build_butterworth_bandpass(
 f_start=0.15,
 f_stop=0.40,
 sample_rate=fs_resample,
 filter_order=2
)

print("LF Digital Filter Coefficients")
print(f"b = {np.round(lf_numerator, 6)}")
print(f"a = {np.round(lf_denominator, 6)}\n")

print("HF Digital Filter Coefficients")
print(f"b = {np.round(hf_numerator, 6)}")
print(f"a = {np.round(hf_denominator, 6)}\n")


# ==========================================================
# Stability check for the LF filter
# ==========================================================
lf_poles = np.roots(lf_denominator)
lf_zeros = np.roots(lf_numerator)

pole_magnitudes = np.abs(lf_poles)
stable_filter = np.all(pole_magnitudes < 1)

print(
 f"Stability: {stable_filter} "
 f"(largest pole magnitude = {np.max(pole_magnitudes):.4f})"
)


# ==========================================================
# Pole-zero plot for LF filter
# ==========================================================
fig, ax = plt.subplots(figsize=(6.5, 6.5))

# Draw unit circle
angle = np.linspace(0, 2 * np.pi, 400)
ax.plot(np.cos(angle), np.sin(angle), linestyle="--", linewidth=1.2, color="black", label="Unit circle")

# Plot zeros
ax.scatter(
 np.real(lf_zeros),
 np.imag(lf_zeros),
 s=85,
 facecolors="none",
 edgecolors="navy",
 linewidths=1.8,
 label="Zeros"
)

# Plot poles
ax.scatter(
 np.real(lf_poles),
 np.imag(lf_poles),
 s=70,
 color="darkred",
 marker="x",
 linewidths=2,
 label="Poles"
)

# Axis lines
ax.axhline(0, color="gray", linewidth=1)
ax.axvline(0, color="gray", linewidth=1)

ax.set_title("Pole-Zero Map of the LF Digital Bandpass Filter")
ax.set_xlabel("Real Axis")
ax.set_ylabel("Imaginary Axis")
ax.set_aspect("equal", adjustable="box")
ax.grid(True, linestyle="--", alpha=0.5)
ax.legend(loc="upper left")

plt.tight_layout()
plt.show()

# 5. Filter Implementation and Validation

## 5.1 Direct-Form Difference Equation Implementation

To apply the designed IIR filter to the data, the filtering operation must be implemented manually. Since the use of built-in functions such as `scipy.signal.lfilter` is restricted for the filter application stage, the filtering process is implemented directly using the **Direct Form I difference equation**.

For an IIR filter, each output sample $y[n]$ depends on both the current and previous input samples, as well as previous output samples. These values are weighted by the numerator coefficients ($b$) and denominator coefficients ($a$) obtained during the filter design stage.

The resulting relationship is expressed as

$$
y[n] = \sum_{i=0}^{M} b_i\,x[n-i] - \sum_{j=1}^{N} a_j\,y[n-j]
$$

This recursive equation is implemented using a loop that processes the signal sample-by-sample, thereby reproducing the behaviour of a digital IIR filter.

![Direct Form I structure](image.png)

---

## 5.2 Validation Using Standard DSP Functions

Although the filter itself must be implemented manually, it is still useful to verify the correctness of the implementation. To do this, the output produced by the custom difference equation is compared against the output obtained using the standard function `scipy.signal.lfilter`.

A single three-minute segment of RRI data is filtered using both approaches. The resulting signals are then compared by calculating the maximum absolute difference between the two outputs. If this difference is negligible (effectively zero), it confirms that the manually implemented difference equation correctly reproduces the behaviour of the standard digital filter.

In [ ]:
import scipy.signal as signal
import numpy as np

def apply_iir_difference_equation(num_coeffs, den_coeffs, input_signal):
 """
 Manual implementation of an IIR filter using the standard difference equation.

 Parameters:
 num_coeffs : numerator coefficients (b array)
 den_coeffs : denominator coefficients (a array), with den_coeffs[0] = 1
 input_signal : signal to be filtered

 Returns:
 filtered_output : output sequence after IIR filtering
 """

 signal_length = len(input_signal)
 filtered_output = np.zeros(signal_length)

 # Compute each output sample one at a time
 for sample_index in range(signal_length):

 # -----------------------------
 # Feedforward contribution
 # -----------------------------
 feedforward_sum = 0.0
 for b_index in range(len(num_coeffs)):
 input_index = sample_index - b_index
 if input_index >= 0:
 feedforward_sum += num_coeffs[b_index] * input_signal[input_index]

 # -----------------------------
 # Feedback contribution
 # -----------------------------
 feedback_sum = 0.0
 for a_index in range(1, len(den_coeffs)): # skip a[0]
 output_index = sample_index - a_index
 if output_index >= 0:
 feedback_sum += den_coeffs[a_index] * filtered_output[output_index]

 # Difference equation
 filtered_output[sample_index] = feedforward_sum - feedback_sum

 return filtered_output


# ==========================================================
# Validate the manual IIR implementation
# ==========================================================
# Filter the detrended / zero-mean RRI signal using:
# 1) the custom loop
# 2) SciPy's reference implementation
lf_output_manual = apply_iir_difference_equation(
 lf_numerator,
 lf_denominator,
 rri_zero_mean
)

lf_output_reference = signal.lfilter(
 lf_numerator,
 lf_denominator,
 rri_zero_mean
)

# Compare the two outputs
absolute_difference = np.abs(lf_output_manual - lf_output_reference)
largest_error = np.max(absolute_difference)

print(f"Maximum absolute difference between manual filter and SciPy reference: {largest_error:.3e}")

if largest_error < 1e-10:
 print("VALIDATION PASSED: The manual difference-equation implementation matches the reference output.")
else:
 print("VALIDATION FAILED: The difference is larger than expected.")

# 6. Drowsiness Detection and Feature Extraction

## 6.1 Rolling Window Signal Processing

After completing the filter design and validation stages, the next step is to apply the filters to the dataset in order to estimate changes in the subject’s physiological state over time. To emulate a real-time monitoring system, a **rolling analysis window** is applied to the `ECGdata.csv` dataset, advancing by one second for each iteration.

**Dataset Adaptation (Window Length Selection):**

The study by Fujiwara (2019) recommends using a **3-minute (180-second) analysis window** when monitoring long driving sessions. However, the provided `ECGdata.csv` dataset contains only approximately **210 seconds** of recorded data. Using a 180-second window would leave very little overlap for the sliding process, making it difficult to observe the gradual transition from wakefulness to the N1 sleep stage, which occurs roughly between **90 seconds and 150 seconds** in the dataset.

To better capture this transition within the limited data duration, the analysis window is reduced to **60 seconds (1 minute)**. Shorter windows are commonly used in **short-term HRV analysis**, and this adjustment allows the algorithm to slide across approximately **150 seconds of data**, providing a clearer view of the evolving physiological changes.

For each rolling window, the following processing steps are performed:

1. **R-peak detection** is used to obtain the sequence of R–R intervals, which are then detrended and resampled at **4 Hz**.
2. The resampled RRI signal is passed through the custom-designed **LF and HF IIR bandpass filters**.
3. The **power of each band** is calculated from the filtered signals. To avoid distortion caused by the filter's transient response, the first **10 seconds** of the filtered signal are discarded.
4. The **LF/HF ratio** is then computed to represent the relative balance between the two frequency components.

---

## 6.2 Comparison with Ground Truth Labels

To assess the reliability of the estimated LF/HF ratio, the calculated values are compared with the expert annotations provided in the dataset. Each sliding window is associated with the **ground-truth state** (`Awake` or `Asleep`) recorded in the CSV file.

The label corresponding to the **center time of the analysis window** is used as the reference state for that window. These annotations are then visualized alongside the computed LF/HF ratio by displaying the labelled states as **shaded regions** in the final trend plot. This allows the estimated physiological indicators to be compared directly with the expert classifications.

In [ ]:
import warnings
import matplotlib.patches as mpatches

warnings.filterwarnings("ignore")

# ==========================================================
# 1) Rolling-window configuration
# ==========================================================
analysis_window_sec = 60 # 1-minute window
window_shift_sec = 1 # move window forward by 1 second

record_length_sec = len(ecg_data) / sampling_rate_ecg
window_count = int((record_length_sec - analysis_window_sec) / window_shift_sec) + 1

window_centers = []
lf_hf_trend = []
reference_labels = []

print(f"Total ECG duration: {record_length_sec:.1f} s")
print(f"Number of rolling windows to evaluate: {window_count}")

# ==========================================================
# 2) Rolling-window processing loop
# ==========================================================
for window_id in range(window_count):

 # ----------------------------------------------
 # Determine current window limits in time
 # ----------------------------------------------
 window_start_sec = window_id * window_shift_sec
 window_end_sec = window_start_sec + analysis_window_sec
 window_mid_sec = window_start_sec + analysis_window_sec / 2

 # Convert time boundaries to sample indices
 start_sample = int(window_start_sec * sampling_rate_ecg)
 end_sample = int(window_end_sec * sampling_rate_ecg)
 mid_sample = int(window_mid_sec * sampling_rate_ecg)

 # Extract ECG segment for the current window
 ecg_window = full_ecg_signal[start_sample:end_sample]
 ecg_window_time = np.arange(len(ecg_window)) / sampling_rate_ecg

 # ----------------------------------------------
 # Detect R-peaks in this ECG segment
 # ----------------------------------------------
 min_peak_gap = int(0.5 * sampling_rate_ecg)
 peak_threshold = np.mean(ecg_window) + 1.5 * np.std(ecg_window)

 peak_indices, _ = find_peaks(
 ecg_window,
 distance=min_peak_gap,
 height=peak_threshold
 )

 # Skip window if too few peaks were found
 if len(peak_indices) < 5:
 continue

 peak_times = ecg_window_time[peak_indices]

 # ----------------------------------------------
 # Form the RRI sequence for this window
 # ----------------------------------------------
 rr_values_ms = np.diff(peak_times) * 1000
 rr_time_values = peak_times[1:]

 # Skip if spline interpolation would fail
 if len(rr_time_values) < 2:
 continue

 # ----------------------------------------------
 # Interpolate and resample at 4 Hz
 # ----------------------------------------------
 resampled_time = np.arange(rr_time_values[0], rr_time_values[-1], 1 / resample_rate)

 if len(resampled_time) < 2:
 continue

 spline_model = CubicSpline(rr_time_values, rr_values_ms)
 rr_uniform = spline_model(resampled_time)

 # Remove mean before filtering
 rr_zero_mean = rr_uniform - np.mean(rr_uniform)

 # ----------------------------------------------
 # Filter the RRI signal in LF and HF bands
 # ----------------------------------------------
 lf_filtered = apply_iir_difference_equation(
 lf_numerator,
 lf_denominator,
 rr_zero_mean
 )

 hf_filtered = apply_iir_difference_equation(
 hf_numerator,
 hf_denominator,
 rr_zero_mean
 )

 # ----------------------------------------------
 # Remove filter startup transient
 # ----------------------------------------------
 transient_samples = int(10 * resample_rate)

 if len(lf_filtered) <= transient_samples or len(hf_filtered) <= transient_samples:
 continue

 lf_valid = lf_filtered[transient_samples:]
 hf_valid = hf_filtered[transient_samples:]

 # ----------------------------------------------
 # Estimate band powers using variance
 # ----------------------------------------------
 lf_band_power = np.var(lf_valid)
 hf_band_power = np.var(hf_valid)

 if hf_band_power == 0:
 continue

 current_ratio = lf_band_power / hf_band_power

 # Use the annotation at the center of the window
 expert_label = ecg_data["Annotation"].iloc[mid_sample]

 # Store result
 window_centers.append(window_mid_sec)
 lf_hf_trend.append(current_ratio)
 reference_labels.append(expert_label)

# ==========================================================
# 3) Plot LF/HF trend together with annotation regions
# ==========================================================
plt.figure(figsize=(12, 6))

# Background shading based on expert labels
state_colors = {
 "Awake": "#8fd19e",
 "Asleep": "#f1a3a3"
}

for idx in range(len(window_centers) - 1):
 state_name = reference_labels[idx]
 shade_color = state_colors.get(state_name, "#d9d9d9")

 plt.axvspan(
 window_centers[idx],
 window_centers[idx + 1],
 color=shade_color,
 alpha=0.35
 )

# Plot the LF/HF ratio trend on top
plt.plot(
 window_centers,
 lf_hf_trend,
 color="midnightblue",
 linewidth=2.2,
 label="Estimated LF/HF ratio"
)

# Optional point markers to make the trend look different
plt.scatter(
 window_centers,
 lf_hf_trend,
 color="midnightblue",
 s=12,
 alpha=0.75
)

# Build custom legend
legend_items = [
 plt.Line2D([0], [0], color="midnightblue", lw=2.2, label="Estimated LF/HF ratio")
]

for label_name in sorted(set(reference_labels)):
 legend_items.append(
 mpatches.Patch(
 color=state_colors.get(label_name, "#d9d9d9"),
 alpha=0.35,
 label=f"Reference state: {label_name}"
 )
 )

plt.title("LF/HF Trend Over Time with Reference Sleep-State Labels")
plt.xlabel("Time [s]")
plt.ylabel("LF/HF Ratio")
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend(handles=legend_items, loc="lower right")
plt.gca().set_facecolor("#f7f7f7")
plt.tight_layout()
plt.show()

# 7. Results and Discussion

The results shown in the graph indicate that the **LF/HF ratio decreases shortly before the dataset annotation marks the transition to the *Asleep* state**. This behaviour is consistent with the physiological expectations described in the literature, where increasing parasympathetic nervous system activity during the onset of sleep leads to a reduction in the LF/HF ratio.

The observed trend suggests that the designed filtering approach successfully captures the changes in autonomic balance reflected in the HRV signal. In the plotted results, the LF/HF ratio shows a noticeable decline prior to the annotated sleep state, followed by a gradual recovery. This pattern aligns with the expected shift from sympathetic dominance during wakefulness toward increased parasympathetic activity as drowsiness develops.

Overall, the results demonstrate that the implemented signal processing pipeline is capable of identifying meaningful physiological changes in the ECG-derived HRV data. The LF/HF ratio therefore serves as a useful indicator for detecting the transition from wakefulness to drowsiness within the analysed dataset.